In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

In [ ]:
# Trial parameters
n_trials = 100
t_star = 960  # True word onset
delta = 30    # ERP detection window ±delta

# Acoustic noise (likelihood)
sigma_l = 30

# Define 3 internal perceptual states
states = {
    0: {'mu_c': 960, 'sigma_p': 30},   # optimal preparation
    1: {'mu_c': 1080, 'sigma_p': 40},  # late prior
    2: {'mu_c': 880, 'sigma_p': 60},   # early & uncertain
}

# Simulate latent state sequence
np.random.seed(42)
z = np.random.choice([0,1,2], size=n_trials, p=[0.4, 0.3, 0.3])


In [ ]:
def posterior_mass(mu_c, sigma_p, t_star, sigma_l, delta):
    # posterior variance and mean
    var_post = 1 / (1 / sigma_p**2 + 1 / sigma_l**2)
    mu_post = var_post * (mu_c / sigma_p**2 + t_star / sigma_l**2)
    sigma_post = np.sqrt(var_post)

    # ERP trigger if posterior mass is within [t_star ± delta]
    lower = t_star - delta
    upper = t_star + delta
    prob = (
        0.5 * (1 + erf((upper - mu_post) / (sigma_post * np.sqrt(2)))) -
        0.5 * (1 + erf((lower - mu_post) / (sigma_post * np.sqrt(2))))
    )
    return prob

from scipy.special import erf

erp_probs = []
erp_labels = []

for i in range(n_trials):
    s = states[z[i]]
    p = posterior_mass(s['mu_c'], s['sigma_p'], t_star, sigma_l, delta)
    erp_probs.append(p)
    erp_labels.append(int(p > 0.75))  # 0 or 1: ERP appears?

erp_probs = np.array(erp_probs)
erp_labels = np.array(erp_labels)


In [ ]:
plt.figure(figsize=(12, 5))
for state_id in states:
    idx = (z == state_id)
    plt.scatter(np.where(idx), erp_probs[idx], label=f"State {state_id}", alpha=0.7)

plt.axhline(0.6, ls='--', color='black', label='ERP Threshold')
plt.xlabel("Trial")
plt.ylabel("Posterior Mass in [t* ± δ]")
plt.title("ERP Probability across Trials and Hidden States")
plt.legend()
plt.show()


In [ ]:
import pandas as pd
df = pd.DataFrame({'State': z, 'ERP': erp_labels})
erp_rate_by_state = df.groupby("State")["ERP"].mean()
print(erp_rate_by_state)